In [13]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

1. we first define the tools
2. then we call our llm models with these tools
model will return the function used and the parameters passed
now we extract these function name and parameters and call the real defined functions
3. these real functions will return an answer
now we pass the role, user query, tools, the answer we get from real function all appending in message list
4. thus now we get final answer

In [14]:
! pip install openai pydantic ddgs

In [15]:
from pydantic import BaseModel, Field
from typing import Optional, Dict, List, Literal, Any
from google.colab import userdata
from openai import OpenAI
import smtplib
from email.mime.text import MIMEText

First make the schema from bottom to top
Then instanciate the tools
then make final tool list

In [16]:
class GetExchangeParam(BaseModel):
  base_currency: str = Field(..., description="The base currency example INR,USD, EUR")
  target_currency: str = Field(..., description="The target currency example INR, USD, EUR")
  date: Optional[str] =Field(None, description="A specific day to reference, in YYYY-MM-DD format.")

class SeacrhInternetParam(BaseModel):
  search_query: str = Field(..., description="The search query")


class FunctionDefination(BaseModel):
  name: str
  description: str
  parameters: Dict[str, Any]

class Tool(BaseModel):
  type: Literal["function"]
  function: FunctionDefination

get_exchange_rate_tool = Tool(
    type="function",
    function=FunctionDefination(
        name="get_exchange_rate",
        description="Get the current exchange rate of a base currency and target currency",
        parameters=GetExchangeParam.model_json_schema()
    )
)

search_internet_tool=Tool(
    type="function",
    function=FunctionDefination(
        name="search_internet",
        description="Search the internet for a query",
        parameters=SeacrhInternetParam.model_json_schema()
    )
)

first_tool: List[Tool] = [
    get_exchange_rate_tool,
    search_internet_tool
]

In [17]:
import json

formatted_tools = []
for tool in first_tool:
    formatted_tools.append(tool.model_dump(mode='json'))

print(json.dumps(formatted_tools, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "get_exchange_rate",
      "description": "Get the current exchange rate of a base currency and target currency",
      "parameters": {
        "properties": {
          "base_currency": {
            "description": "The base currency example INR,USD, EUR",
            "title": "Base Currency",
            "type": "string"
          },
          "target_currency": {
            "description": "The target currency example INR, USD, EUR",
            "title": "Target Currency",
            "type": "string"
          },
          "date": {
            "anyOf": [
              {
                "type": "string"
              },
              {
                "type": "null"
              }
            ],
            "default": null,
            "description": "A specific day to reference, in YYYY-MM-DD format.",
            "title": "Date"
          }
        },
        "required": [
          "base_currency",
          "target_

Now we call or llm and pass the tools to get the tools used and the parameters used

In [18]:
input_message = input("user: ")
message=[
    {
        'role':'system',
        'content':'''
        you are a helpful assistant.

        you have tools:
        1. get_exchange_rate
        2. search_internet

        ''',
    },
    {
        'role':'user',
        'content':input_message
    }
]

Client = OpenAI(api_key=userdata.get('raj_api_key'))
for i in range(3):
  response = Client.chat.completions.create(
    model="gpt-4o",
    messages=message,
    tools=formatted_tools,
    tool_choice="auto"
)


print(response.usage)
for i in range(0,len(response.choices[0].message.tool_calls)):
  print(response.choices[0].message.tool_calls[i].function.name)
  print(response.choices[0].message.tool_calls[i].function.arguments)

user: conver usd to inr and latest cricket news
CompletionUsage(completion_tokens=56, prompt_tokens=196, total_tokens=252, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))
get_exchange_rate
{"base_currency": "USD", "target_currency": "INR"}
search_internet
{"search_query": "latest cricket news"}


now we write our tools


In [19]:
import requests
from bs4 import BeautifulSoup

#Tool 1
def get_exchange_rate_tool(base_currency: str, target_currency: str, date: str="latest")-> float:
  url = f"https://cdn.jsdelivr.net/npm/@fawazahmed0/currency-api@{date}/v1/currencies/{base_currency.lower()}.json"
  response = requests.get(url)

  if response.status_code == 200:
      data = response.json()
      return data.get(base_currency.lower(), {}).get(target_currency.lower(), None)
  else:
      raise Exception(f"Failed to fetch exchange rate: {response.status_code}")
#Tool 2
def search_internet_tool(search_query: str)-> list:
  url = f"https://news.google.com/rss/search?q={search_query}"
  response = requests.get(url)
  soup = BeautifulSoup(response.content, "xml")
  items = soup.find_all("item")
  results = []
  for item in items[:5]:
    title = item.title.text
    results.append(title)
  return results



now we call our function then append the tool(function) result

In [20]:
import json
import inspect

tools_message=[]

#we store the responses and tools used
response_message= response.choices[0].message
tool_calls= response.choices[0].message.tool_calls
for i in range(5):
#now we do a check if tool calls happend or not
  if tool_calls:
  #we connect the tool calls fubnctions
    available_functions = {
            "get_exchange_rate": get_exchange_rate_tool,
            "search_internet": search_internet_tool
        }

    message.append(response_message)

  #we call our functions
    for tool_call in tool_calls:
      function_name=tool_call.function.name
      function_to_call=available_functions[function_name]
      function_args=json.loads(tool_call.function.arguments)

    #function signature
      sig=inspect.signature(function_to_call)
      bound=sig.bind_partial(**function_args)
      bound.apply_defaults()

      result=function_to_call(*bound.args, **bound.kwargs)

    # Convert the result to a string, preferably JSON if it's a list or dict
      if isinstance(result, (list, dict)):
        final_result = json.dumps(result)
      else:
        final_result = str(result)

    #now we add the result in tools message
      message.append(
        {
            "tool_call_id": tool_call.id,
            'role':'tool',
            'name':function_name,
            'content':final_result
        }
      )

now we do our final llm call and see the output

In [21]:
response_final=Client.chat.completions.create(
    model="gpt-4o",
    messages=message,

)

print("AI: ", response_final.choices[0].message.content)

AI:  The current exchange rate from USD to INR is approximately 95.05.

Here are some of the latest cricket news headlines:

1. **Sixers unveil former Aussie cricket star as new Big Bash coach** - Fox Sports
2. **Cricket News Today**: Mark Wood targets a summer comeback after injury setback with England future on the line - readcricket.com
3. **"India-Jamaica story written in runs"**: External Affairs Minister Jaishankar - lokmattimes.com
4. **India in The ICC Men’s T20 World Cup Final**: Five Fast Facts You Need to Know - heavy.com
5. **India T20 World Cup victory**: Fans celebrate across the country - BBC
